In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mahotas as mh
import imutils
import nd2
import cv2
import os
import pandas as pd
from scipy.signal import find_peaks
pwd = os.getcwd()

In [ ]:
IMAGE_SCALE = 0.5605 # pixel/micron
SCALE_BAR = 200 # micron
SCALE_BAR_MARGIN = 50 # pixels
SCALE_BAR_THICKNESS = 20 # pixels

# Get the group number from the user
sample_ID = input("Enter sample number: ")

# Define paths
path_results = ['./cropped_images']
folder_address = ['./raw_images']

# Create results folder if it doesn't exist
# path_results = os.path.join(pwd,results_address[0] + '_results', 'sliced_images')
if not os.path.exists(path_results[0]):
    os.makedirs(path_results[0])
    
# Create DataFrame to store perimeter and area data
data_columns = ['Sample ID', 'Image', 'Sample number', 'Organoid number', 'Day', 'Perimeter', 'Area']
perimeter_data = pd.DataFrame(columns=data_columns)

In [ ]:
# Get list of image filenames
# for .tif images
all_imgs_names = [f for f in os.listdir(folder_address[0]) \
              if (os.path.splitext(os.path.join(pwd,f))[1] == '.tif')]

# for .jpg images
# all_imgs_names = [f for f in os.listdir(folder_address[0]) \
#                   if (os.path.splitext(os.path.join(pwd,f))[1] == '.jpg')]

print(len(all_imgs_names))

In [ ]:
for img_num in range(len(all_imgs_names)):
    print(all_imgs_names[img_num])
    
    # Read the color image
    img_color = cv2.imread(os.path.join(folder_address[0], 
                                        all_imgs_names[img_num]), 
                           cv2.IMREAD_UNCHANGED)
    
    # Convert color image to grayscale
    # bf_img = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY) # chatgpt
    bf_img = cv2.imread(os.path.join(folder_address[0],
                                     all_imgs_names[img_num]), 
                            cv2.IMREAD_GRAYSCALE)
    
    # Normalize image intensities
    # print(np.max(bf_img))
    bf_img = bf_img * int(255/np.max(bf_img))
    plt.figure()
    plt.imshow(bf_img, 'gray')
    plt.show()

    mask_img = cv2.adaptiveThreshold(bf_img,255,cv2.ADAPTIVE_THRESH_MEAN_C,\
            cv2.THRESH_BINARY_INV,699,3)
    # th3 = cv2.adaptiveThreshold(bf_img,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,\
    #         cv2.THRESH_BINARY,699,3)

    plt.figure()
    plt.imshow(mask_img, 'gray')
    plt.show()
    
    # Label image regions
    labeled_img, n_img = mh.label(mask_img)
    labeled_img, n_img = mh.labeled.filter_labeled(labeled_img, 
                                                   remove_bordering=True, 
                                                   min_size=10000)
    plt.figure()
    plt.imshow(labeled_img, 'gray')
    # plt.show()
    nn = 1
    
    for obj in np.unique(labeled_img):
        # if the label is zero, we are examining the 'background', so simply ignore it
        if obj == 0:
            continue
            
        # otherwise, allocate memory for the label region and draw it on the mask
        mask = np.zeros(labeled_img.shape, dtype="uint8")
        mask[labeled_img == obj] = 255

        # detect contours in the green mask and grab the largest one
        cnts = cv2.findContours(mask.copy(), 
                                cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
        cnts = imutils.grab_contours(cnts)
        c = max(cnts, key=cv2.contourArea)

        
        ## Detailed perimeter
        # Approximate the contour with a more detailed shape
        epsilon = 0.0001 * cv2.arcLength(c, True)  # Adjust the epsilon value as needed
        approx = cv2.approxPolyDP(c, epsilon, True)
        scale_factor = 1 / IMAGE_SCALE  # Convert pixels/micron to microns/pixel
        
        # Calculate perimeter of the approximated contour
        perimeter_pixels = cv2.arcLength(approx, True)
        perimeter_microns = perimeter_pixels * scale_factor
        print("Perimeter of object", obj, ":", perimeter_microns, "microns")
        
        area = cv2.contourArea(c) * (1 / (IMAGE_SCALE ** 2))  # Convert pixels^2 to square microns

        name_raw = str.split(all_imgs_names[img_num], '.')[0]
        sample_number = str.split(name_raw, ' ')[0]
        temp_str = str.split(name_raw, ' ')[1]
        organoid_number = str.split(temp_str, '_')[1]
        day = str.split(temp_str, '_')[0]
        
        # Add data to DataFrame
        perimeter_data.loc[len(perimeter_data)] = {'Sample ID': sample_ID,
                                                   'Image': all_imgs_names[img_num],
                                                   'Sample number': sample_number, 
                                                   'Organoid number': organoid_number,
                                                   'Day': int(day), 
                                                   'Perimeter': perimeter_microns, 
                                                   'Area': area}

        
        # ## LESS Detailed perimeter
        # # measure perimeter
        # scale_factor = 1 / IMAGE_SCALE  # Convert pixels/micron to microns/pixel
        # # Calculate perimeter of the contour
        # perimeter_pixels = cv2.arcLength(c, True)
        # perimeter_microns = perimeter_pixels * scale_factor
        # print("Perimeter of object", obj, ":", perimeter_microns, "microns")
           
            
        # Draw a line on the perimeter of the object
        colored_img = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
        cv2.drawContours(colored_img, [approx], -1, (255, 0, 0), 2)  # Draw contour in blue color with thickness 2

        # Display the new color image with the line drawn on the perimeter
        # plt.figure()
        plt.imshow(colored_img)
        plt.title('Color Image with Perimeter')
        plt.show()
        
               
        # Save the cropped image
        cv2.imwrite(os.path.join(path_results[0], os.path.splitext(os.path.basename(os.path.join(folder_address[0], 
                                                                                                 all_imgs_names[img_num])))[0]+'_'+str(nn)+'.tif'), colored_img)
        
        nn += 1

In [ ]:
# Save perimeter data to Excel
excel_filename = f"perimeter_data_sample_{sample_ID}.xlsx"
perimeter_data['Day'] = perimeter_data['Day'].astype(int)
perimeter_data.sort_values('Day', ascending=True, inplace=True)
perimeter_data.to_excel(excel_filename, index=False)
print(f"Perimeter data saved to {excel_filename}")